# TP4 – Gestion du cache avec Redis

## Configuration du notebook

In [ ]:
!pip install redis pymongo pandas numpy --quiet

In [ ]:
# Valeurs à remplacer
## Redis
REDIS_HOST = "...redis.io"
REDIS_PORT = 14031
REDIS_USER = "default"
REDIS_PWD = "..."

## MongoDB
MONGODB_URI = 'mongodb://localhost:27017'

In [ ]:
import json
import os
import time

import numpy as np
import pandas as pd
import redis
from pymongo import MongoClient

# Redis
redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    username=REDIS_USER,
    password=REDIS_PWD,
    decode_responses=True,
)

print(redis_client.ping())

In [ ]:
# MongoDB
MONGODB_URI = 'mongodb://localhost:27017'
client = MongoClient()
restaurants = client["tp"]["restaurants"]

print(restaurants.count_documents({}))

# 3. Utilisation de Redis avec Python

## 3.2 Charger et interroger des restaurants

In [ ]:
redis_client.flushdb()

curseur = restaurants.find(
    {"borough": "Bronx", "grades.score": {"$type": "number"}},
    {"_id": 0, "restaurant_id": 1, "name": 1, "cuisine": 1, "borough": 1, "grades.score": 1},
).sort("restaurant_id", 1).limit(500)

documents = []
for doc in curseur:
    scores = [g["score"] for g in doc["grades"] if g.get("score") is not None]
    documents.append({
        "id": doc["restaurant_id"],
        "nom": doc["name"],
        "cuisine": doc["cuisine"],
        "quartier": doc["borough"],
        "note": round(sum(scores) / len(scores), 2),
    })

len(documents)